In [20]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import polars as pl
import numpy as np
import re
from datetime import date

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(100)

device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

Device: mps


In [21]:
tech_path = "../data/model_staging/tech_modeling_table.parquet"
df_tech = pl.read_parquet(tech_path)
print(f"Tech table shape: {df_tech.shape}")

Tech table shape: (24048, 239)


In [22]:
df_tech.schema

Schema([('symbol', String),
        ('earnings_date', Date),
        ('entry_price', Float64),
        ('target_return', Float64),
        ('target_class', Int64),
        ('car_3day', Float64),
        ('car_t2_t10', Float64),
        ('max_high', Float64),
        ('min_high', Float64),
        ('max_day', Int64),
        ('min_day', Int64),
        ('open_pct_t-10', Float64),
        ('high_pct_t-10', Float64),
        ('low_pct_t-10', Float64),
        ('volume_rel_t-10', Float64),
        ('rsi_t-10', Float64),
        ('macd_t-10', Float64),
        ('macd_hist_t-10', Float64),
        ('roc_t-10', Float64),
        ('ema50_pct_t-10', Float64),
        ('ema200_pct_t-10', Float64),
        ('ema50_200_pct_t-10', Float64),
        ('adx_t-10', Float64),
        ('atr_t-10', Float64),
        ('bb_width_t-10', Float64),
        ('bb_pct_b_t-10', Float64),
        ('sigma_t-10', Float64),
        ('obv_zscore_t-10', Float64),
        ('vwap_pct_t-10', Float64),
        ('vix_close_t

In [23]:
fund_path = "../data/model_staging/fundamentalIndicators/modeling_fundamentals.parquet"
df_fund = pl.read_parquet(fund_path)
df_fund = df_fund.select([
    "symbol", pl.col("reportedDate").alias("earnings_date"),
    "eps_growth_qoq", "revenue_growth_qoq",
    "gross_margin", "gross_margin_qoq",
    "debt_to_equity", "debt_to_equity_qoq",
    "fcf_margin", "fcf_margin_qoq",
    "roe", "roe_qoq"
])

df_finbert = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_finbert = df_finbert.select([
    pl.col("symbol"), pl.col("reportedDate").alias("earnings_date"),
    "pos_prob", "neg_prob"
])

df_nz = pl.read_parquet("../data/model_staging/nz_sentiment.parquet")
df_nz = df_nz.select([
    pl.col("symbol"), pl.col("reportedDate").alias("earnings_date"),
    "overall_sentiment_score_pre", "ticker_sentiment_score_pre",
    "overall_sentiment_score_post", "ticker_sentiment_score_post",
])

df_model = df_tech.join(df_fund, on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_finbert, on=["symbol", "earnings_date"], how="left", suffix="_fb")
df_model = df_model.join(df_nz, on=["symbol", "earnings_date"], how="left", suffix="_nz")

df_sector_raw = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_sector = df_sector_raw.select(["symbol", "sector"]).unique()
df_sector = df_sector.with_columns(pl.col("sector").fill_null("Unknown"))
sectors = sorted(df_sector["sector"].unique().to_list())
df_sector = df_sector.with_columns([
    (pl.col("sector") == s).cast(pl.Int8).alias(f"sector_{s.replace(' ', '_')}")
    for s in sectors
]).drop("sector")
df_model = df_model.join(df_sector, on="symbol", how="left")

print(f"Final shape: {df_model.shape}")

Final shape: (24048, 267)


In [24]:
df_model

symbol,earnings_date,entry_price,target_return,target_class,car_3day,car_t2_t10,max_high,min_high,max_day,min_day,open_pct_t-10,high_pct_t-10,low_pct_t-10,volume_rel_t-10,rsi_t-10,macd_t-10,macd_hist_t-10,roc_t-10,ema50_pct_t-10,ema200_pct_t-10,ema50_200_pct_t-10,adx_t-10,atr_t-10,bb_width_t-10,bb_pct_b_t-10,sigma_t-10,obv_zscore_t-10,vwap_pct_t-10,vix_close_t-10,open_pct_t-9,high_pct_t-9,low_pct_t-9,volume_rel_t-9,rsi_t-9,macd_t-9,macd_hist_t-9,roc_t-9,ema50_pct_t-9,ema200_pct_t-9,ema50_200_pct_t-9,adx_t-9,atr_t-9,bb_width_t-9,bb_pct_b_t-9,sigma_t-9,obv_zscore_t-9,vwap_pct_t-9,vix_close_t-9,open_pct_t-8,…,obv_zscore_t0,vwap_pct_t0,vix_close_t0,open_pct_t+1,high_pct_t+1,low_pct_t+1,volume_rel_t+1,rsi_t+1,macd_t+1,macd_hist_t+1,roc_t+1,ema50_pct_t+1,ema200_pct_t+1,ema50_200_pct_t+1,adx_t+1,atr_t+1,bb_width_t+1,bb_pct_b_t+1,sigma_t+1,obv_zscore_t+1,vwap_pct_t+1,vix_close_t+1,eps_growth_qoq,revenue_growth_qoq,gross_margin,gross_margin_qoq,debt_to_equity,debt_to_equity_qoq,fcf_margin,fcf_margin_qoq,roe,roe_qoq,pos_prob,neg_prob,overall_sentiment_score_pre,ticker_sentiment_score_pre,overall_sentiment_score_post,ticker_sentiment_score_post,sector_Basic_Materials,sector_Communication_Services,sector_Consumer_Cyclical,sector_Consumer_Defensive,sector_Energy,sector_Financial_Services,sector_Healthcare,sector_Industrials,sector_Real_Estate,sector_Technology,sector_Unknown,sector_Utilities
str,date,f64,f64,i64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""LYB""",2020-07-31,42.475105,0.183004,2,-0.099336,0.084218,50.248203,44.539732,7,2,0.008175,0.017518,-0.006131,-0.466738,56.244829,0.011621,0.002661,2.299903,0.060346,-0.007481,-0.063967,10.51737,0.040626,0.100177,0.794481,0.020628,-0.306764,0.047946,25.68,0.019012,0.032036,-0.002545,-0.432572,51.672713,0.010835,0.001317,-2.12451,0.032652,-0.031804,-0.062418,10.024723,0.041154,0.096452,0.520329,0.021298,-0.411784,0.022934,24.459999,-0.008227,…,-0.392894,-0.063225,24.459999,0.014808,0.02704,-0.000322,0.344674,38.427014,-0.004488,-0.011676,-6.991045,-0.049149,-0.096821,-0.050136,11.404466,0.040181,0.135342,-0.067169,0.021281,-0.61579,-0.063568,24.280001,-0.537415,-0.259941,0.117562,0.034029,3.0,0.040398,0.126938,0.142684,0.042539,0.02315,0.28713,0.269593,0.229063,0.116009,0.224658,0.23145,1,0,0,0,0,0,0,0,0,0,0,0
"""LYB""",2023-10-27,76.03109,0.036976,1,0.019794,-0.022796,78.842392,76.131806,10,3,0.003776,0.014674,-0.0041,-0.323695,42.163044,-0.013228,-0.001008,-2.133043,-0.023657,0.015104,0.039701,30.334723,0.022799,0.054506,0.389251,0.013491,0.762318,-0.018973,19.32,0.001927,0.007173,-0.006745,-0.421522,45.343562,-0.012117,0.000006,-0.042806,-0.01535,0.022865,0.038811,30.418959,0.022076,0.054363,0.536082,0.012991,0.814194,-0.007018,17.209999,-0.022797,…,0.800155,-0.017975,21.27,0.008057,0.010265,-0.02064,0.147614,41.6783,-0.014836,-0.001394,-3.008253,-0.033177,-0.008061,0.025978,34.210997,0.022954,0.069164,0.289271,0.012625,0.899396,-0.01437,19.75,0.008197,0.030953,0.136282,-0.003248,1.78281,-0.044118,0.119718,0.023754,0.056418,0.001202,0.427493,0.362845,0.131983,0.14841,0.271922,0.11083,1,0,0,0,0,0,0,0,0,0,0,0
"""LYB""",2021-04-30,76.533699,0.082507,2,0.013232,0.061157,82.848248,77.569582,6,2,0.00623,0.014877,-0.006137,-0.281581,58.323214,0.007363,0.001179,1.894857,0.051681,0.227514,0.167193,10.782928,0.027548,0.050157,0.983224,0.011011,1.457469,0.027006,16.25,0.005037,0.007929,-0.0125,-0.414743,57.146784,0.008037,0.001466,2.86919,0.046278,0.220804,0.166806,10.50057,0.027123,0.053627,0.851412,0.0104,1.387482,0.026602,17.290001,0.019973,…,1.320815,-0.012261,18.610001,-0.02707,0.010568,-0.033281,0.455346,55.25571,0.004287,0.000273,0.625,0.042566,0.205374,0.156161,8.139

In [25]:
exclude_cols = ["symbol", "earnings_date", "target_return", "target_class",
                "entry_price", "max_high", "min_high", "max_day", "min_day",
                "car_3day", "car_t2_t10"]
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

time_cols = [c for c in feature_cols if re.match(r".+_t[+-]?\d+$", c)]
static_cols = [c for c in feature_cols if c not in time_cols]

bases = sorted(set(re.sub(r"_t[+-]?\d+$", "", c) for c in time_cols))
steps = sorted({int(re.search(r"_t([+-]?\d+)$", c).group(1)) for c in time_cols})
n_steps, n_bases, n_static = len(steps), len(bases), len(static_cols)
print(f"Time steps: {steps}  ({n_steps} steps x {n_bases} bases = {n_steps * n_bases})")
print(f"Static features: {n_static}")

Time steps: [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1]  (12 steps x 19 bases = 228)
Static features: 28


In [26]:
df_model.head(10).write_clipboard()

In [29]:
import polars as pl
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- derive year and quarter from earnings_date ---
df_model = df_model.with_columns(
    pl.col("earnings_date").cast(pl.Date).alias("earnings_date")
).with_columns([
    pl.col("earnings_date").dt.year().alias("year"),
    pl.col("earnings_date").dt.quarter().alias("quarter"),
])

df_model = df_model.with_columns(
    (pl.col("year") * 10 + pl.col("quarter")).alias("year_quarter")
)

# initial training cutoff: everything through 2020 is the starting train set
initial_cutoff = 2020 * 10 + 4  # i.e. 2020 Q4

# test quarters: everything from 2021 Q1 onward
test_quarters = (
    df_model
    .filter(pl.col("year_quarter") > initial_cutoff)
    .select("year_quarter")
    .unique()
    .sort("year_quarter")
    .to_series()
    .to_list()
)

results = []
all_y_test, all_preds = [], []

for yq in test_quarters:
    train_df = df_model.filter(pl.col("year_quarter") <= max(initial_cutoff, yq - 1))
    test_df  = df_model.filter(pl.col("year_quarter") == yq)

    if train_df.height == 0 or test_df.height == 0:
        continue

    X_train_full = train_df.select(feature_cols).to_pandas()
    y_train_full = train_df["target_return"].to_pandas()

    X_test = test_df.select(feature_cols).to_pandas()
    y_test = test_df["target_return"].to_pandas()

    # carve last 10% of train chronologically for early stopping validation
    n_val = max(1, int(len(X_train_full) * 0.1))
    X_fit, X_val = X_train_full.iloc[:-n_val], X_train_full.iloc[-n_val:]
    y_fit, y_val = y_train_full.iloc[:-n_val], y_train_full.iloc[-n_val:]

    model = XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=30,
        eval_metric="rmse",
    )

    model.fit(
        X_fit, y_fit,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds) if len(y_test) > 1 else float("nan")

    results.append({
        "year_quarter": yq,
        "n_train": len(X_train_full),
        "n_test": len(X_test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

    all_y_test.extend(y_test.tolist())
    all_preds.extend(preds.tolist())

    print(f"Q{y - q}: train={len(X_train_full):5d}  test={len(X_test):4d}  "
          f"MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

results_df = pl.DataFrame(results)
print(results_df)

pooled_r2 = r2_score(all_y_test, all_preds)
pooled_mae = mean_absolute_error(all_y_test, all_preds)
pooled_rmse = np.sqrt(mean_squared_error(all_y_test, all_preds))

print(f"\nAvg R2 across quarters:  {results_df['R2'].mean():.4f}")
print(f"Pooled R2 (2021-latest): {pooled_r2:.4f}")
print(f"Pooled MAE:              {pooled_mae:.4f}")
print(f"Pooled RMSE:             {pooled_rmse:.4f}")

Q20211: train=13169  test= 492  MAE=0.0353  RMSE=0.0477  R2=0.1807
Q20212: train=13661  test= 493  MAE=0.0264  RMSE=0.0389  R2=-0.1363
Q20213: train=14154  test= 493  MAE=0.0239  RMSE=0.0328  R2=0.0767
Q20214: train=14647  test= 492  MAE=0.0249  RMSE=0.0373  R2=0.0703
Q20221: train=15139  test= 494  MAE=0.0366  RMSE=0.0516  R2=0.0681
Q20222: train=15633  test= 495  MAE=0.0437  RMSE=0.0529  R2=-0.5309
Q20223: train=16128  test= 493  MAE=0.0270  RMSE=0.0347  R2=0.1451
Q20224: train=16621  test= 493  MAE=0.0388  RMSE=0.0540  R2=0.1037
Q20231: train=17114  test= 492  MAE=0.0261  RMSE=0.0342  R2=0.1182
Q20232: train=17606  test= 498  MAE=0.0256  RMSE=0.0395  R2=0.1295
Q20233: train=18104  test= 496  MAE=0.0209  RMSE=0.0325  R2=-0.1431
Q20234: train=18600  test= 498  MAE=0.0314  RMSE=0.0415  R2=-0.0094
Q20241: train=19098  test= 496  MAE=0.0266  RMSE=0.0411  R2=0.2146
Q20242: train=19594  test= 501  MAE=0.0267  RMSE=0.0405  R2=0.1314
Q20243: train=20095  test= 499  MAE=0.0272  RMSE=0.0366  R

In [10]:
importances = (
    pl.DataFrame({"feature": feature_cols, "importance": model.feature_importances_})
    .sort("importance", descending=True)
    .head(20)
)
print(importances)

shape: (20, 2)
┌────────────────┬────────────┐
│ feature        ┆ importance │
│ ---            ┆ ---        │
│ str            ┆ f32        │
╞════════════════╪════════════╡
│ atr_t+1        ┆ 0.037881   │
│ atr_t0         ┆ 0.030841   │
│ atr_t-1        ┆ 0.025872   │
│ sigma_t-1      ┆ 0.018879   │
│ atr_t-5        ┆ 0.01528    │
│ ema200_pct_t-1 ┆ 0.014815   │
│ bb_pct_b_t-8   ┆ 0.013889   │
│ volume_rel_t-4 ┆ 0.012014   │
│ adx_t-6        ┆ 0.011812   │
│ adx_t-5        ┆ 0.01147    │
│ vix_close_t+1  ┆ 0.01054    │
│ adx_t-8        ┆ 0.009265   │
│ rsi_t-3        ┆ 0.009253   │
│ bb_width_t-2   ┆ 0.009181   │
│ rsi_t-4        ┆ 0.008884   │
│ ema200_pct_t+1 ┆ 0.008851   │
│ bb_width_t-4   ┆ 0.008728   │
│ adx_t-7        ┆ 0.008246   │
│ ema50_pct_t-6  ┆ 0.00824    │
│ vix_close_t0   ┆ 0.008096   │
└────────────────┴────────────┘


In [11]:
atr_cols = [f"atr_t-{i}" for i in range(1, 11)]
df_model = df_model.with_columns(
    (pl.mean_horizontal(atr_cols) * np.sqrt(9)).alias("sigma_pre_9day")
).with_columns(
    (pl.col("target_return") / pl.col("sigma_pre_9day")).alias("target_return_scaled")
)

In [14]:
print(train_df.select(
    pl.col("sigma_pre_9day").min().alias("min"),
    pl.col("sigma_pre_9day").max().alias("max"),
    pl.col("sigma_pre_9day").is_null().sum().alias("nulls"),
    (pl.col("sigma_pre_9day") == 0).sum().alias("zeros"),
))

print(train_df.select(
    pl.col("target_return_scaled").is_infinite().sum().alias("infs"),
    pl.col("target_return_scaled").is_nan().sum().alias("nans"),
    pl.col("target_return_scaled").is_null().sum().alias("nulls"),
))

shape: (1, 4)
┌──────────┬──────────┬───────┬───────┐
│ min      ┆ max      ┆ nulls ┆ zeros │
│ ---      ┆ ---      ┆ ---   ┆ ---   │
│ f64      ┆ f64      ┆ u32   ┆ u32   │
╞══════════╪══════════╪═══════╪═══════╡
│ 0.005131 ┆ 0.790147 ┆ 0     ┆ 0     │
└──────────┴──────────┴───────┴───────┘
shape: (1, 3)
┌──────┬──────┬───────┐
│ infs ┆ nans ┆ nulls │
│ ---  ┆ ---  ┆ ---   │
│ u32  ┆ u32  ┆ u32   │
╞══════╪══════╪═══════╡
│ 0    ┆ 48   ┆ 0     │
└──────┴──────┴───────┘


In [15]:
df_model = df_model.filter(
    pl.col("target_return_scaled").is_finite() & pl.col("target_return_scaled").is_not_null()
)

train_df = df_model.filter(pl.col("year") < 2024)
test_df  = df_model.filter(pl.col("year") >= 2024)

X_train = train_df.select(feature_cols).to_pandas()
y_train = train_df["target_return_scaled"].to_pandas()

X_test = test_df.select(feature_cols).to_pandas()
y_test = test_df["target_return_scaled"].to_pandas()

In [16]:
# 1. Add the scaled target
atr_cols = [f"atr_t-{i}" for i in range(1, 11)]
df_model = df_model.with_columns(
    (pl.mean_horizontal(atr_cols) * np.sqrt(9)).alias("sigma_pre_9day")
).with_columns(
    (pl.col("target_return") / pl.col("sigma_pre_9day")).alias("target_return_scaled")
)

# 2. Re-slice train/test from the UPDATED df_model
train_df = df_model.filter(pl.col("year") < 2024)
test_df  = df_model.filter(pl.col("year") >= 2024)

# 3. Features stay the same, but y now comes from the scaled column
X_train = train_df.select(feature_cols).to_pandas()
y_train = train_df["target_return_scaled"].to_pandas()

X_test = test_df.select(feature_cols).to_pandas()
y_test = test_df["target_return_scaled"].to_pandas()

model_scaled = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric="rmse",
)

model_scaled.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

preds_scaled = model_scaled.predict(X_test)

mae_scaled = mean_absolute_error(y_test, preds_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, preds_scaled))
r2_scaled = r2_score(y_test, preds_scaled)

print(f"MAE (scaled):  {mae_scaled:.4f}")
print(f"RMSE (scaled): {rmse_scaled:.4f}")
print(f"R2 (scaled):   {r2_scaled:.4f}")

[0]	validation_0-rmse:0.53468
[50]	validation_0-rmse:0.53362
[100]	validation_0-rmse:0.53240
[150]	validation_0-rmse:0.53205
[200]	validation_0-rmse:0.53152
[216]	validation_0-rmse:0.53170
MAE (scaled):  0.3868
RMSE (scaled): 0.5314
R2 (scaled):   0.0074


In [17]:
importances = (
    pl.DataFrame({"feature": feature_cols, "importance": model.feature_importances_})
    .sort("importance", descending=True)
    .head(20)
)
print(importances)

shape: (20, 2)
┌────────────────┬────────────┐
│ feature        ┆ importance │
│ ---            ┆ ---        │
│ str            ┆ f32        │
╞════════════════╪════════════╡
│ atr_t+1        ┆ 0.037881   │
│ atr_t0         ┆ 0.030841   │
│ atr_t-1        ┆ 0.025872   │
│ sigma_t-1      ┆ 0.018879   │
│ atr_t-5        ┆ 0.01528    │
│ ema200_pct_t-1 ┆ 0.014815   │
│ bb_pct_b_t-8   ┆ 0.013889   │
│ volume_rel_t-4 ┆ 0.012014   │
│ adx_t-6        ┆ 0.011812   │
│ adx_t-5        ┆ 0.01147    │
│ vix_close_t+1  ┆ 0.01054    │
│ adx_t-8        ┆ 0.009265   │
│ rsi_t-3        ┆ 0.009253   │
│ bb_width_t-2   ┆ 0.009181   │
│ rsi_t-4        ┆ 0.008884   │
│ ema200_pct_t+1 ┆ 0.008851   │
│ bb_width_t-4   ┆ 0.008728   │
│ adx_t-7        ┆ 0.008246   │
│ ema50_pct_t-6  ┆ 0.00824    │
│ vix_close_t0   ┆ 0.008096   │
└────────────────┴────────────┘


In [18]:
vol_cols = [c for c in feature_cols if re.match(r"(atr|sigma|bb_width|vix_close)_t", c)]
non_vol_cols = [c for c in feature_cols if c not in vol_cols]

In [19]:
import re

# Split features into volatility-related vs everything else
vol_cols = [c for c in feature_cols if re.match(r"(atr|sigma|bb_width|vix_close)_t", c)]
non_vol_cols = [c for c in feature_cols if c not in vol_cols]

print(f"Vol features: {len(vol_cols)}")
print(f"Non-vol features: {len(non_vol_cols)}")

# reuse train_df/test_df from the ORIGINAL (unscaled) split — target_return, not scaled
X_train_full = train_df.select(feature_cols).to_pandas()
y_train = train_df["target_return"].to_pandas()
X_test_full = test_df.select(feature_cols).to_pandas()
y_test = test_df["target_return"].to_pandas()

def run_model(cols, label):
    Xtr = X_train_full[cols]
    Xte = X_test_full[cols]
    m = XGBRegressor(
        n_estimators=500, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, early_stopping_rounds=30, eval_metric="rmse",
    )
    m.fit(Xtr, y_train, eval_set=[(Xte, y_test)], verbose=False)
    preds = m.predict(Xte)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    print(f"{label:20s}  R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  (n_features={len(cols)})")
    return m, r2

model_vol, r2_vol = run_model(vol_cols, "Vol-only")
model_nonvol, r2_nonvol = run_model(non_vol_cols, "Non-vol (rest)")
model_full, r2_full = run_model(feature_cols, "Full model")

Vol features: 48
Non-vol features: 208
Vol-only              R2=0.1635  RMSE=0.0450  MAE=0.0294  (n_features=48)
Non-vol (rest)        R2=0.1556  RMSE=0.0452  MAE=0.0297  (n_features=208)
Full model            R2=0.1754  RMSE=0.0447  MAE=0.0292  (n_features=256)
